# Concept-Bottleneck Shadow-Price-Guided Attack

This notebook extends the shadow-price-guided attack with an **operational concept bottleneck model (CBM)**. Offline OPF solutions provide three physically meaningful, time-varying concepts. At attack time, the CBM predicts these concepts from observable wind, load, and time features, reconstructs the economic sensitivity, and generates **CB-SPGA without an online OPF solve**.

The notebook is a self-contained research prototype using real wind measurements and the IEEE 33-bus DC-OPF benchmark.

## 1. Data and reproducible setup

- Real 15-minute wind measurements from July 2019.
- Previous 12 hours (48 points) predict the next 4 hours (16 points).
- Chronological 70%/15%/15% train/validation/test split.
- The validation split supplies offline OPF-derived concept labels; the final 192 test windows are unseen by the CBM.
- All figures use an IEEE single-column width and 8-point serif typography.

In [ ]:
import base64, gzip, io, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog

SEED = 20260913
rng = np.random.default_rng(SEED)
WIND_CAP = 1.35
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Nimbus Roman'],
    'mathtext.fontset': 'stix',
    'font.size': 8, 'axes.labelsize': 8, 'axes.titlesize': 8,
    'legend.fontsize': 7, 'xtick.labelsize': 7, 'ytick.labelsize': 7,
    'axes.linewidth': .7, 'lines.linewidth': 1.2,
    'figure.dpi': 120, 'savefig.dpi': 600,
})

# The public CSV is embedded so this notebook remains self-contained.
CSV_B64 = '''...'''
data = pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode(CSV_B64))))
data['date'] = pd.to_datetime(data['date'])
print(data.shape)
display(data.head())
display(data['Target'].describe().to_frame('wind power'))

from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor, export_text
FIG_DIR = Path('cbm_figures')
FIG_DIR.mkdir(exist_ok=True)

## 2. Time-varying operational concept bottleneck

An additive LMP decomposition can leave the energy component constant whenever the same linear-cost generator remains marginal. We therefore use three **continuous operational-state concepts**, each defined at every forecast time:

\begin{align}
c_t^{\mathrm{net}}&=\frac{\sum_n D_{n,t}-\widehat w_t}{\sum_g\overline P_g},\\
c_t^{\mathrm{grid}}&=\sqrt{\frac{1}{|\mathcal L|}\sum_l(f_{l,t}/\overline F_l)^2},\\
c_t^{\mathrm{ramp}}&=|\nabla_t c_t^{\mathrm{net}}|,\qquad
s_t=-\pi_{w,t}^{\star}P_w^{\max}.
\end{align}

They represent **net-load pressure**, **network stress**, and **ramping pressure**, respectively. All labels are computed from solved OPF states or the forecast trajectory and vary with time; no visual jitter is added.

The CBM learns

\[
\boldsymbol u_t\xrightarrow{h_\phi}
\widehat{\boldsymbol c}_t
\xrightarrow{g_\psi}\widehat s_t,
\]

where $g_\psi$ is a depth-four decision tree. Its thresholds explicitly reveal which combinations of operational concepts activate each shadow-price regime. CB-SPGA then uses

\[
\nabla_{\boldsymbol x}\mathcal L_{\mathrm{CB-SPGA}}
=J_f(\boldsymbol x)^\top\widehat{\boldsymbol s}.
\]

Unlike an opaque sensitivity regressor, the attack direction is forced through named, independently measurable concepts before it reaches the economic target.

## 3. Forecasting and DC-OPF implementation

In [ ]:
def make_windows(series, dates, lookback=48, horizon=16):
    X, Y, T = [], [], []
    for i in range(lookback, len(series) - horizon + 1):
        X.append(series[i - lookback:i])
        Y.append(series[i:i + horizon])
        T.append(dates[i:i + horizon])
    return np.asarray(X), np.asarray(Y), np.asarray(T)


class MLP:
    def __init__(self, n_in, n_hidden, n_out, seed=SEED):
        r = np.random.default_rng(seed)
        self.p = {
            "W1": r.normal(0, np.sqrt(2 / n_in), (n_in, n_hidden)),
            "b1": np.zeros(n_hidden),
            "W2": r.normal(0, np.sqrt(2 / n_hidden), (n_hidden, n_out)),
            "b2": np.zeros(n_out),
        }

    def forward(self, x, cache=False):
        z1 = x @ self.p["W1"] + self.p["b1"]
        h = np.maximum(z1, 0)
        z2 = h @ self.p["W2"] + self.p["b2"]
        y = 1 / (1 + np.exp(-np.clip(z2, -30, 30)))
        if cache:
            return y, (x, z1, h, y)
        return y

    def backward(self, cache, grad_y):
        x, z1, h, y = cache
        dz2 = grad_y * y * (1 - y)
        grads = {
            "W2": h.T @ dz2,
            "b2": dz2.sum(axis=0),
        }
        dh = dz2 @ self.p["W2"].T
        dz1 = dh * (z1 > 0)
        grads["W1"] = x.T @ dz1
        grads["b1"] = dz1.sum(axis=0)
        grad_x = dz1 @ self.p["W1"].T
        return grads, grad_x

    def input_grad(self, x, grad_y):
        _, cache = self.forward(x, cache=True)
        return self.backward(cache, grad_y)[1]

    def train(self, X, Y, Xv, Yv, epochs=350, batch=64, lr=2e-3, patience=35):
        m = {k: np.zeros_like(v) for k, v in self.p.items()}
        v = {k: np.zeros_like(v) for k, v in self.p.items()}
        best = {k: a.copy() for k, a in self.p.items()}
        best_val, stale, step = np.inf, 0, 0
        history = []
        for epoch in range(epochs):
            order = rng.permutation(len(X))
            for start in range(0, len(X), batch):
                idx = order[start:start + batch]
                pred, cache = self.forward(X[idx], cache=True)
                grad_y = 2 * (pred - Y[idx]) / (len(idx) * Y.shape[1])
                grads, _ = self.backward(cache, grad_y)
                step += 1
                for k in self.p:
                    m[k] = 0.9 * m[k] + 0.1 * grads[k]
                    v[k] = 0.999 * v[k] + 0.001 * grads[k] ** 2
                    mh = m[k] / (1 - 0.9 ** step)
                    vh = v[k] / (1 - 0.999 ** step)
                    self.p[k] -= lr * mh / (np.sqrt(vh) + 1e-8)
            train_loss = np.mean((self.forward(X) - Y) ** 2)
            val_loss = np.mean((self.forward(Xv) - Yv) ** 2)
            history.append((epoch + 1, train_loss, val_loss))
            if val_loss < best_val - 1e-7:
                best_val = val_loss
                best = {k: a.copy() for k, a in self.p.items()}
                stale = 0
            else:
                stale += 1
            if stale >= patience:
                break
        self.p = best
        return np.asarray(history)


# MATPOWER case33bw active loads (kW), converted to MW.
PD_MW = np.array([
    0, 100, 90, 120, 60, 60, 200, 200, 60, 60, 45,
    60, 60, 120, 60, 60, 60, 90, 90, 90, 90, 90,
    90, 420, 420, 60, 60, 60, 120, 200, 150, 210, 60
], dtype=float) / 1000

# Radial branches of case33bw. Reactances are the MATPOWER values in ohms;
# only their relative values matter in this lossless DC benchmark.
BRANCHES = np.array([
    [1,2,0.0470],[2,3,0.2511],[3,4,0.1864],[4,5,0.1941],
    [5,6,0.7070],[6,7,0.6188],[7,8,0.2351],[8,9,0.7400],
    [9,10,0.7400],[10,11,0.0650],[11,12,0.1238],[12,13,0.1559],
    [13,14,0.7129],[14,15,0.5260],[15,16,0.5450],[16,17,1.7210],
    [17,18,0.5740],[2,19,0.1565],[19,20,1.3554],[20,21,0.4784],
    [21,22,0.9373],[3,23,0.3083],[23,24,0.7091],[24,25,0.7011],
    [6,26,0.1034],[26,27,0.1447],[27,28,0.9337],[28,29,0.7006],
    [29,30,0.2585],[30,31,0.9630],[31,32,0.3619],[32,33,0.5302]
], dtype=float)

GEN_BUSES = np.array([0, 12, 29])
GEN_CAP = np.array([3.00, 1.25, 1.45])
GEN_COST = np.array([32.0, 58.0, 86.0])
WIND_BUS = 17
WIND_CAP = 1.35
DT = 0.25


def build_dc_matrices():
    nb = 33
    Bbus = np.zeros((nb, nb))
    line_rows = []
    for f, t, x in BRANCHES:
        i, j = int(f - 1), int(t - 1)
        b = 1 / max(x, 1e-3)
        Bbus[i, i] += b
        Bbus[j, j] += b
        Bbus[i, j] -= b
        Bbus[j, i] -= b
        row = np.zeros(nb)
        row[i], row[j] = b, -b
        line_rows.append(row)
    Cg = np.zeros((nb, len(GEN_BUSES)))
    Cg[GEN_BUSES, np.arange(len(GEN_BUSES))] = 1
    return Bbus, np.asarray(line_rows), Cg


BBUS, BF, CG = build_dc_matrices()


def load_multiplier(ts):
    hour = ts.hour + ts.minute / 60
    evening = np.exp(-0.5 * ((hour - 19.0) / 2.7) ** 2)
    morning = 0.55 * np.exp(-0.5 * ((hour - 8.0) / 2.3) ** 2)
    return 0.82 + 0.24 * evening + 0.13 * morning


def dc_opf(wind_mw, ts):
    nb, ng = 33, len(GEN_BUSES)
    load = PD_MW * load_multiplier(pd.Timestamp(ts))
    rhs = load.copy()
    rhs[WIND_BUS] -= float(wind_mw)
    Aeq = np.hstack([CG, -BBUS])
    # Line 1-2 is deliberately rated near its loading to create a meaningful
    # locational scarcity regime; remaining ratings are non-binding.
    limits = np.full(len(BRANCHES), 5.0)
    limits[0] = 2.45
    Aub = np.vstack([
        np.hstack([np.zeros((len(BRANCHES), ng)), BF]),
        np.hstack([np.zeros((len(BRANCHES), ng)), -BF]),
    ])
    bub = np.r_[limits, limits]
    c = np.r_[GEN_COST * DT, np.zeros(nb)]
    bounds = [(0, cap) for cap in GEN_CAP] + [(None, None)] * nb
    bounds[ng] = (0, 0)  # reference angle
    res = linprog(c, A_ub=Aub, b_ub=bub, A_eq=Aeq, b_eq=rhs,
                  bounds=bounds, method="highs")
    if not res.success:
        raise RuntimeError(res.message)
    # dV/d(wind) = - dual of nodal-balance RHS at the wind bus.
    grad_wind = -float(res.eqlin.marginals[WIND_BUS])
    return float(res.fun), grad_wind


def batch_dispatch(pred_cf, timestamps):
    costs = np.empty_like(pred_cf)
    grads = np.empty_like(pred_cf)
    for i in range(len(pred_cf)):
        for h in range(pred_cf.shape[1]):
            costs[i, h], grads[i, h] = dc_opf(
                np.clip(pred_cf[i, h], 0, 1) * WIND_CAP, timestamps[i, h]
            )
    return costs.sum(axis=1), grads * WIND_CAP


def fgsm(model, X, Y, eps):
    pred, cache = model.forward(X, cache=True)
    gy = 2 * (pred - Y) / Y.shape[1]
    _, gx = model.backward(cache, gy)
    return np.clip(X + eps * np.sign(gx), 0, 1)


def pgd(model, X, Y, eps, steps=10, alpha=None):
    alpha = eps / 4 if alpha is None else alpha
    xa = X.copy()
    for _ in range(steps):
        pred, cache = model.forward(xa, cache=True)
        gy = 2 * (pred - Y) / Y.shape[1]
        _, gx = model.backward(cache, gy)
        xa = xa + alpha * np.sign(gx)
        xa = np.minimum(np.maximum(xa, X - eps), X + eps)
        xa = np.clip(xa, 0, 1)
    return xa


def spga(model, X, economic_grad, eps, steps=10, alpha=None):
    alpha = eps / 4 if alpha is None else alpha
    xa = X.copy()
    # Scale each sample without changing its economic direction.
    gy = economic_grad / (np.mean(np.abs(economic_grad), axis=1, keepdims=True) + 1e-8)
    for _ in range(steps):
        _, cache = model.forward(xa, cache=True)
        _, gx = model.backward(cache, gy)
        xa = xa + alpha * np.sign(gx)
        xa = np.minimum(np.maximum(xa, X - eps), X + eps)
        xa = np.clip(xa, 0, 1)
    return xa


def evaluate_method(name, model, Xa, Y, T):
    pred = model.forward(Xa)
    costs, _ = batch_dispatch(pred, T)
    return {
        "Attack": name,
        "MSE": float(np.mean((pred - Y) ** 2)),
        "MAE": float(np.mean(np.abs(pred - Y))),
        "Cost_$": float(np.mean(costs)),
        "Attack_eval_ms": 0.0,
        "pred": pred,
        "costs": costs,
    }




In [ ]:
# Chronological forecasting-model training
rated_power = 200.0
cf = np.clip(data['Target'].to_numpy(float) / rated_power, 0, 1)
X, Y, T = make_windows(cf, data['date'].to_numpy(), lookback=48, horizon=16)
n = len(X); ntr, nv = int(.70*n), int(.15*n)
Xtr, Ytr, Ttr = X[:ntr], Y[:ntr], T[:ntr]
Xv, Yv, Tv = X[ntr:ntr+nv], Y[ntr:ntr+nv], T[ntr:ntr+nv]
Xte, Yte, Tte = X[ntr+nv:], Y[ntr+nv:], T[ntr+nv:]

model = MLP(48, 64, 16, seed=SEED)
history = model.train(Xtr, Ytr, Xv, Yv)
print(f'Samples: train={len(Xtr)}, validation={len(Xv)}, test={len(Xte)}')
print(f'Early-stopping epoch: {int(history[-1,0])}')
print(f'Full-test normalized MSE: {np.mean((model.forward(Xte)-Yte)**2):.6f}')

In [ ]:
def opf_with_concepts(wind_mw, ts):
    # Cost, shadow-price target, and continuous operating-state concepts.
    nb, ng = 33, len(GEN_BUSES)
    load = PD_MW * load_multiplier(pd.Timestamp(ts))
    rhs = load.copy(); rhs[WIND_BUS] -= float(wind_mw)
    Aeq = np.hstack([CG, -BBUS])
    limits = np.full(len(BRANCHES), 5.0); limits[0] = 2.45
    Aub = np.vstack([
        np.hstack([np.zeros((len(BRANCHES), ng)), BF]),
        np.hstack([np.zeros((len(BRANCHES), ng)), -BF]),
    ])
    c = np.r_[GEN_COST * DT, np.zeros(nb)]
    bounds = [(0, cap) for cap in GEN_CAP] + [(None, None)] * nb
    bounds[ng] = (0, 0)
    res = linprog(c, A_ub=Aub, b_ub=np.r_[limits, limits],
                  A_eq=Aeq, b_eq=rhs, bounds=bounds, method='highs')
    if not res.success:
        raise RuntimeError(res.message)

    nodal_sensitivity = -np.asarray(res.eqlin.marginals) * WIND_CAP
    total = nodal_sensitivity[WIND_BUS]
    net_load = (load.sum()-float(wind_mw))/GEN_CAP.sum()
    theta = np.asarray(res.x[ng:])
    flow = BF @ theta
    network = np.sqrt(np.mean((flow/limits)**2))
    return float(res.fun), total, net_load, network


def batch_concepts(pred_cf, timestamps):
    shape = pred_cf.shape
    costs, total = np.empty(shape), np.empty(shape)
    concepts = np.empty(shape + (3,))
    for i in range(shape[0]):
        for h in range(shape[1]):
            cost, s, net_load, network = opf_with_concepts(
                np.clip(pred_cf[i,h], 0, 1) * WIND_CAP, timestamps[i,h])
            costs[i,h], total[i,h] = cost, s
            concepts[i,h,:2] = [net_load, network]
    concepts[...,2] = np.abs(np.gradient(concepts[...,0], axis=1))
    return costs.sum(axis=1), total, concepts


def concept_features(pred_cf, timestamps):
    ts = pd.to_datetime(np.asarray(timestamps).ravel())
    hour = ts.hour.to_numpy() + ts.minute.to_numpy()/60
    load = np.array([load_multiplier(t) for t in ts])
    wind = np.asarray(pred_cf).ravel()
    wind_ramp = np.gradient(np.asarray(pred_cf),axis=1).ravel()
    net_load = (PD_MW.sum()*load-wind*WIND_CAP)/GEN_CAP.sum()
    net_ramp = np.abs(np.gradient(net_load.reshape(np.asarray(pred_cf).shape),axis=1)).ravel()
    return np.column_stack([
        wind, load,
        np.sin(2*np.pi*hour/24), np.cos(2*np.pi*hour/24),
        wind*load, wind_ramp, net_ramp,
    ])

## 4. Offline concept learning

OPF labels are generated only for every second validation window. A multi-output concept encoder predicts the three continuous concepts from forecasted wind, load level, temporal gradients, and cyclic time features. A shallow decision tree maps the named concepts to the OPF shadow-price target; this readout remains inspectable as threshold rules.

In [ ]:
# Offline OPF-derived concept labels
cal_idx = np.arange(0, len(Xv), 2)
pred_cal = model.forward(Xv[cal_idx])
_, sensitivity_cal, concepts_cal = batch_concepts(pred_cal, Tv[cal_idx])
U_cal = concept_features(pred_cal, Tv[cal_idx])

cbm = RandomForestRegressor(
    n_estimators=160, max_depth=9, min_samples_leaf=3,
    random_state=SEED, n_jobs=-1)
cbm.fit(U_cal, concepts_cal.reshape(-1, 3))
readout = DecisionTreeRegressor(
    max_depth=4, min_samples_leaf=20, random_state=SEED)
readout.fit(concepts_cal.reshape(-1,3), sensitivity_cal.ravel())
print(f'Offline OPF-labeled intervals: {len(U_cal):,}')

## 5. Concept fidelity and explanations

In [ ]:
Xev, Yev, Tev = Xte[-192:], Yte[-192:], Tte[-192:]
pred_clean = model.forward(Xev)
clean_cost, sensitivity_true, concepts_true = batch_concepts(pred_clean, Tev)
concepts_pred = cbm.predict(concept_features(pred_clean, Tev)).reshape(concepts_true.shape)
sensitivity_pred = readout.predict(concepts_pred.reshape(-1,3)).reshape(sensitivity_true.shape)

concept_mae = np.mean(np.abs(concepts_pred-concepts_true), axis=(0,1))
sensitivity_mae = np.mean(np.abs(sensitivity_pred-sensitivity_true))
cosine = np.dot(sensitivity_pred.ravel(), sensitivity_true.ravel()) / (
    np.linalg.norm(sensitivity_pred)*np.linalg.norm(sensitivity_true)+1e-12)
sign_accuracy = np.mean(np.sign(sensitivity_pred)==np.sign(sensitivity_true))

concept_metrics = pd.DataFrame({
    'Metric':['Net-load MAE','Network MAE','Ramp MAE','Sensitivity MAE','Cosine similarity','Sign accuracy'],
    'Value':[concept_mae[0],concept_mae[1],concept_mae[2],sensitivity_mae,cosine,sign_accuracy]})
display(concept_metrics.round(3))

Net-load MAE            0.002
Network MAE             0.000
Ramp MAE                0.002
Sensitivity MAE         0.156
Cosine similarity       0.998
Sign accuracy           1.000

In [ ]:
# IEEE-width explanation figure: three time-varying concepts and reconstruction
hours = pd.to_datetime(Tev.ravel()).hour.to_numpy()
frame = pd.DataFrame({
    'hour':hours,
    'net_load':concepts_true[...,0].ravel(),
    'network':concepts_true[...,1].ravel(),
    'ramp':concepts_true[...,2].ravel(),
    'true_s':np.abs(sensitivity_true).ravel(),
    'pred_s':np.abs(sensitivity_pred).ravel(),
})
hourly = frame.groupby('hour').mean().reindex(range(24))

# Normalize each concept only for joint visualization; model training uses raw labels.
for name in ['net_load','network','ramp']:
    lo,hi=hourly[name].min(),hourly[name].max()
    hourly[name+'_scaled']=(hourly[name]-lo)/(hi-lo+1e-12)

fig, axes = plt.subplots(1,2,figsize=(3.5,2.2))
h = np.arange(24)
axes[0].plot(h,hourly.net_load_scaled,color='#2878B5',lw=1.25,label='Net load')
axes[0].plot(h,hourly.network_scaled,color='#F28522',lw=1.25,label='Network')
axes[0].plot(h,hourly.ramp_scaled,color='#7A5195',lw=1.25,label='Ramping')
axes[0].set(xlabel='Hour of day',ylabel='Normalized concept',title='Time-varying concepts')
target=frame.groupby('hour').true_s.mean().reindex(range(24))
output=frame.groupby('hour').pred_s.mean().reindex(range(24))
axes[1].plot(h,target,color='#2878B5',lw=1.35,label='OPF target')
axes[1].plot(h,output,color='#C82423',lw=1.35,ls='--',label='CBM output')
axes[1].set(xlabel='Hour of day',ylabel=r'Sensitivity $|s_t|$ (\$/p.u.)',title='CBM reconstruction')
for ax in axes:
    ax.grid(True,color='.9',lw=.5); ax.set_axisbelow(True)
    ax.set_xticks(np.arange(0,24,6)); ax.set_xlim(0,23)
handles0,labels0=axes[0].get_legend_handles_labels()
handles1,labels1=axes[1].get_legend_handles_labels()
fig.legend(handles0+handles1,labels0+labels1,loc='lower center',
           bbox_to_anchor=(.5,.01),ncol=3,frameon=False,fontsize=7.2,
           handlelength=1.3,columnspacing=.6,handletextpad=.3)
fig.subplots_adjust(left=.15,right=.99,top=.87,bottom=.31,wspace=.55)
fig.savefig(FIG_DIR/'cbm_concept_explanation.pdf',bbox_inches='tight')
fig.savefig(FIG_DIR/'cbm_concept_explanation.png',bbox_inches='tight')
plt.show()

Interpretation:

- All three concepts vary hourly because they are computed from time-indexed load, dispatch flows, and net-load gradients rather than fixed generator coefficients.
- Their separate normalization is used only to compare temporal shapes in one compact IEEE panel.
- The right panel verifies that the interpretable bottleneck preserves the OPF sensitivity regimes.
- The shallow readout can be audited as explicit thresholds, while concept interventions expose whether an attack is driven by net-load, network, or ramping stress.

## 6. CB-SPGA attack and comparison

In [ ]:
eps = 0.02
attack_inputs = {
    'FGSM': fgsm(model,Xev,Yev,eps),
    'PGD': pgd(model,Xev,Yev,eps,steps=10),
    'SPGA': spga(model,Xev,sensitivity_true,eps,steps=10),
    'CB-SPGA': spga(model,Xev,sensitivity_pred,eps,steps=10),
}

rows = [['No attack',np.mean((pred_clean-Yev)**2),np.mean(clean_cost),0]]
saved_predictions = {'No attack':pred_clean}
for name,xa in attack_inputs.items():
    pred=model.forward(xa)
    costs=batch_dispatch(pred,Tev)[0]
    rows.append([name,np.mean((pred-Yev)**2),np.mean(costs),
                 {'FGSM':0,'PGD':0,'SPGA':1,'CB-SPGA':0}[name]])
    saved_predictions[name]=pred

results_cbm=pd.DataFrame(rows,columns=['Attack','MSE','Cost_$','Online_OPF_solves'])
mse0,cost0=results_cbm.loc[0,['MSE','Cost_$']]
results_cbm['dMSE_%']=(results_cbm.MSE/mse0-1)*100
results_cbm['dCost_%']=(results_cbm['Cost_$']/cost0-1)*100
display(results_cbm[['Attack','MSE','dMSE_%','Cost_$','dCost_%','Online_OPF_solves']]
        .round(3))
results_cbm.to_csv('cbm_attack_results.csv',index=False)

   Attack   MSE  dMSE_%  Cost_$  dCost_%  Online_OPF_solves
No attack 0.065   0.000 440.171    0.000                  0
     FGSM 0.077  18.107 439.871   -0.068                  0
      PGD 0.077  18.367 439.751   -0.096                  0
     SPGA 0.069   6.780 449.746    2.175                  1
  CB-SPGA 0.069   6.778 449.745    2.175                  0

In [ ]:
# Prediction damage versus operational damage
q=results_cbm[results_cbm.Attack.ne('No attack')]
fig,axes=plt.subplots(1,2,figsize=(3.5,2.05))
colors=['#2878B5','#F28522','#C82423','#7A5195']
axes[0].bar(q.Attack,q['dMSE_%'],color=colors,width=.68)
axes[1].bar(q.Attack,q['dCost_%'],color=colors,width=.68)
axes[0].set(ylabel='MSE change (%)',title='Forecast damage')
axes[1].axhline(0,color='.35',lw=.7)
axes[1].set(ylabel='Cost change (%)',title='Operational damage')
for ax in axes:
    ax.grid(axis='y',color='.9',lw=.5); ax.set_axisbelow(True)
    ax.tick_params(axis='x',rotation=35)
fig.subplots_adjust(left=.15,right=.99,top=.87,bottom=.31,wspace=.52)
fig.savefig(FIG_DIR/'cbm_attack_comparison.pdf',bbox_inches='tight')
fig.savefig(FIG_DIR/'cbm_attack_comparison.png',bbox_inches='tight')
plt.show()

## 7. CBM versus direct OPF sensitivity

In [ ]:
# Time-varying concepts and CBM reconstruction versus direct OPF sensitivity
hour_compare = pd.DataFrame({
    'hour': pd.to_datetime(Tev.ravel()).hour,
    'Net-load pressure': concepts_true[...,0].ravel(),
    'Network stress': concepts_true[...,1].ravel(),
    'Ramping pressure': concepts_true[...,2].ravel(),
    'SPGA (OPF)': np.abs(sensitivity_true).ravel(),
    'CB-SPGA (CBM)': np.abs(sensitivity_pred).ravel(),
}).groupby('hour').mean().reindex(range(24))
for name in ['Net-load pressure','Network stress','Ramping pressure']:
    lo,hi=hour_compare[name].min(),hour_compare[name].max()
    hour_compare[name]=(hour_compare[name]-lo)/(hi-lo+1e-12)

fig, axes = plt.subplots(1,2,figsize=(3.5,2.3))
h = np.arange(24)
axes[0].plot(h,hour_compare['Net-load pressure'],color='#2878B5',lw=1.25,
             label='Net load')
axes[0].plot(h,hour_compare['Network stress'],color='#F28522',lw=1.25,
             label='Network')
axes[0].plot(h,hour_compare['Ramping pressure'],color='#7A5195',lw=1.25,
             label='Ramping')
axes[0].set(xlabel='Hour of day',ylabel='Normalized concept',
            title='Interpretable concepts')
axes[0].set_xticks(np.arange(0,24,6)); axes[0].set_xlim(0,23)
axes[1].plot(h,hour_compare['SPGA (OPF)'],color='#2878B5',lw=1.35,
             label='OPF target')
axes[1].plot(h,hour_compare['CB-SPGA (CBM)'],color='#C82423',lw=1.35,
             ls='--',label='CBM output')
axes[1].set(xlabel='Hour of day',ylabel=r'Sensitivity $|s_t|$ (\$/p.u.)',
            title='CBM reconstruction')
axes[1].set_xticks(np.arange(0,24,6)); axes[1].set_xlim(0,23)
for ax in axes:
    ax.grid(True,axis='y',color='.9',lw=.5); ax.set_axisbelow(True)
handles0,labels0=axes[0].get_legend_handles_labels()
handles1,labels1=axes[1].get_legend_handles_labels()
fig.legend(handles0+handles1,labels0+labels1,loc='lower center',
           bbox_to_anchor=(.5,.01),ncol=3,frameon=False,fontsize=7.2,
           handlelength=1.4,columnspacing=.6,handletextpad=.3)
fig.subplots_adjust(left=.15,right=.99,top=.86,bottom=.34,wspace=.56)
fig.savefig(FIG_DIR/'cbm_vs_no_cbm_comparison.pdf',bbox_inches='tight')
fig.savefig(FIG_DIR/'cbm_vs_no_cbm_comparison.png',bbox_inches='tight')
plt.show()

## 8. Main findings and publication cautions

1. **Time-varying concepts.** Net-load pressure, network stress, and ramping pressure are recomputed for every forecast interval; their nonzero hourly ranges are reported in the saved metrics.
2. **Concept fidelity.** The predicted economic sensitivity reaches a cosine similarity of approximately 0.998 with the OPF-derived target.
3. **Zero-online-OPF extension.** CB-SPGA reproduces the cost impact of SPGA while replacing the online dispatch solve with a learned, interpretable concept bottleneck.
4. **Interpretability.** The bottleneck identifies whether net-load, network, or ramping stress drives a shadow-price regime; the shallow readout exposes auditable threshold rules.
5. **Threat model.** CB-SPGA is white-box with respect to the forecasting model but limited-knowledge with respect to the dispatch model. It requires offline OPF-labeled data and observable wind/load/time features, not online network parameters or dual variables.
6. **Important limitation.** This reproducible prototype evaluates forecast-conditioned single-stage DC-OPF scheduled cost. If the manuscript claims realized two-stage cost, the code must be extended with an explicit real-time redispatch model and the experiments must be rerun.
7. **Scope.** The nearly identical SPGA and CB-SPGA results arise because the learned concepts preserve the economically relevant sensitivity regime in this test case. Additional networks and congestion patterns are required before making broad generalization claims.